# Gold Admission Star

Build the admission-level Gold fact and supporting diagnosis/procedure bridges.

Outputs:
- `fact_admission`
- `bridge_admission_diagnosis`
- `bridge_admission_procedure`

Grain:
- `fact_admission`: one row per `hadm_id`
- diagnosis bridge: one row per admission diagnosis
- procedure bridge: one row per admission procedure

Cleaning already completed in Silver is reused rather than repeated here.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_SCHEMA = "cdr_silver.processed_mimic_iv"
GOLD_SCHEMA = "cdr_gold.analytics_mimic_iv"

print(f"Silver source: {SILVER_SCHEMA}")
print(f"Gold target:   {GOLD_SCHEMA}")

In [0]:
# Admission-level sources

admissions = spark.table(f"{SILVER_SCHEMA}.admissions")
patients = spark.table(f"{SILVER_SCHEMA}.patients")
transfers = spark.table(f"{SILVER_SCHEMA}.transfers")
services = spark.table(f"{SILVER_SCHEMA}.services")
icustays = spark.table(f"{SILVER_SCHEMA}.icustays")

diagnoses = spark.table(f"{SILVER_SCHEMA}.icd_diagnoses")
procedures = spark.table(f"{SILVER_SCHEMA}.icd_procedures")

# Persisted Gold dimensions
dim_patient = spark.table(f"{GOLD_SCHEMA}.dim_patient")
dim_date = spark.table(f"{GOLD_SCHEMA}.dim_date")
dim_careunit = spark.table(f"{GOLD_SCHEMA}.dim_careunit")
dim_service = spark.table(f"{GOLD_SCHEMA}.dim_service")
dim_diagnosis = spark.table(f"{GOLD_SCHEMA}.dim_diagnosis")
dim_procedure = spark.table(f"{GOLD_SCHEMA}.dim_procedure")

print("Admission star source tables loaded.")

In [0]:
# Admission transfer summary
# Grain: one row per hadm_id

# All transfer history linked to an admission
journey_events = (
    transfers
    .filter(F.col("hadm_id").isNotNull())
)

# Valid careunit records only for careunit-based fields
valid_careunit_events = (
    journey_events
    .filter(F.col("careunit_valid_flag") == 1)
)

first_window = Window.partitionBy("hadm_id").orderBy(
    F.col("intime").asc(),
    F.col("transfer_id").asc()
)

last_window = Window.partitionBy("hadm_id").orderBy(
    F.col("intime").desc(),
    F.col("transfer_id").desc()
)

# First valid careunit
first_careunit = (
    valid_careunit_events
    .withColumn("_rn", F.row_number().over(first_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("careunit").alias("first_careunit")
    )
)

# Last valid careunit
last_careunit = (
    valid_careunit_events
    .withColumn("_rn", F.row_number().over(last_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("careunit").alias("last_careunit")
    )
)

# True transfer count
transfer_metrics = (
    journey_events
    .groupBy("hadm_id")
    .agg(
        F.sum(
            F.when(
                F.col("eventtype") == "transfer",
                1
            ).otherwise(0)
        ).alias("transfer_count")
    )
)

# Distinct valid careunits
careunit_metrics = (
    valid_careunit_events
    .groupBy("hadm_id")
    .agg(
        F.countDistinct("careunit")
        .alias("distinct_careunit_count")
    )
)

transfer_summary = (
    transfer_metrics
    .join(careunit_metrics, "hadm_id", "left")
    .join(first_careunit, "hadm_id", "left")
    .join(last_careunit, "hadm_id", "left")
)

display(
    transfer_summary.agg(
        F.count("*").alias("admissions_with_transfer_history"),
        F.sum(
            F.when(F.col("first_careunit").isNull(), 1).otherwise(0)
        ).alias("admissions_without_valid_careunit"),
        F.min("transfer_count").alias("min_transfer_count"),
        F.max("transfer_count").alias("max_transfer_count")
    )
)

In [0]:
# Admission service summary
# Grain: one row per hadm_id

service_events = (
    services
    .filter(F.col("hadm_id").isNotNull())
)

# Prefer the true initial row where prev_service is null
initial_service_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(
        F.col("transfertime").asc(),
        F.col("prev_service").isNotNull().asc(),
        F.col("curr_service").asc()
    )
)

final_service_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(
        F.col("transfertime").desc(),
        F.col("curr_service").asc()
    )
)

initial_service = (
    service_events
    .withColumn("_rn", F.row_number().over(initial_service_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("curr_service").alias("initial_service")
    )
)

final_service = (
    service_events
    .withColumn("_rn", F.row_number().over(final_service_window))
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("curr_service").alias("final_service")
    )
)

service_metrics = (
    service_events
    .groupBy("hadm_id")
    .agg(
        F.sum(
            F.when(
                F.col("prev_service").isNotNull()
                & (F.col("prev_service") != F.col("curr_service")),
                1
            ).otherwise(0)
        ).alias("service_change_count")
    )
)

service_summary = (
    service_metrics
    .join(initial_service, "hadm_id", "left")
    .join(final_service, "hadm_id", "left")
)

display(
    service_summary.agg(
        F.count("*").alias("admissions_with_services"),
        F.sum(
            F.when(F.col("initial_service").isNull(), 1).otherwise(0)
        ).alias("missing_initial_service"),
        F.sum(
            F.when(F.col("final_service").isNull(), 1).otherwise(0)
        ).alias("missing_final_service"),
        F.min("service_change_count").alias("min_service_changes"),
        F.max("service_change_count").alias("max_service_changes")
    )
)

In [0]:
# ICU summary
# Grain: one row per hadm_id with at least one ICU stay

icu_summary = (
    icustays
    .filter(F.col("hadm_id").isNotNull())
    .groupBy("hadm_id")
    .agg(
        F.countDistinct("stay_id").alias("icu_stay_count"),

        F.sum(
            F.when(
                F.col("los_availability_flag") == 0,
                1
            ).otherwise(0)
        ).alias("_missing_los_stays"),

        F.sum("los").alias("_known_icu_los_days")
    )
    .withColumn(
        "icu_flag",
        F.lit(1)
    )
    .withColumn(
        "icu_los_complete",
        F.col("_missing_los_stays") == 0
    )
    .withColumn(
        "total_icu_los_days",
        F.when(
            F.col("_missing_los_stays") == 0,
            F.round(F.col("_known_icu_los_days"), 2)
        ).otherwise(
            F.lit(None).cast("double")
        )
    )
    .drop(
        "_missing_los_stays",
        "_known_icu_los_days"
    )
)

display(
    icu_summary.agg(
        F.count("*").alias("admissions_with_icu"),
        F.sum(
            F.when(~F.col("icu_los_complete"), 1).otherwise(0)
        ).alias("incomplete_icu_los"),
        F.min("icu_stay_count").alias("min_icu_stays"),
        F.max("icu_stay_count").alias("max_icu_stays")
    )
)

In [0]:
# Build admission diagnosis/procedure bridges
# These preserve the many-to-many relationships between admissions
# and ICD diagnosis/procedure dimensions.

bridge_admission_diagnosis = (
    diagnoses
    .filter(F.col("hadm_id").isNotNull())
    .join(
        dim_diagnosis.select(
            "diagnosis_key",
            "icd_code",
            "icd_version"
        ),
        on=["icd_code", "icd_version"],
        how="left"
    )
    .withColumn(
        "is_primary_diagnosis",
        F.col("seq_num") == 1
    )
    .select(
        "hadm_id",
        "diagnosis_key",
        "seq_num",
        "is_primary_diagnosis"
    )
)

bridge_admission_procedure = (
    procedures
    .filter(F.col("hadm_id").isNotNull())
    .join(
        dim_procedure.select(
            "procedure_key",
            "icd_code",
            "icd_version"
        ),
        on=["icd_code", "icd_version"],
        how="left"
    )
    .select(
        "hadm_id",
        "procedure_key",
        "seq_num"
    )
)

# Admission-level summaries derived from the bridges
diagnosis_summary = (
    bridge_admission_diagnosis
    .groupBy("hadm_id")
    .agg(
        F.count("*").alias("diagnosis_count"),
        F.max(
            F.when(
                F.col("is_primary_diagnosis"),
                F.col("diagnosis_key")
            )
        ).alias("primary_diagnosis_key")
    )
)

procedure_summary = (
    bridge_admission_procedure
    .groupBy("hadm_id")
    .agg(
        F.count("*").alias("procedure_count")
    )
)

# Validate dimension key matches
display(
    spark.createDataFrame([
        {
            "bridge": "diagnosis",
            "rows": bridge_admission_diagnosis.count(),
            "missing_dimension_keys": bridge_admission_diagnosis
                .filter(F.col("diagnosis_key").isNull())
                .count()
        },
        {
            "bridge": "procedure",
            "rows": bridge_admission_procedure.count(),
            "missing_dimension_keys": bridge_admission_procedure
                .filter(F.col("procedure_key").isNull())
                .count()
        }
    ])
)

In [0]:
# Deterministic primary diagnosis selection
# Prefer ICD-10 when multiple seq_num = 1 records exist.

primary_diagnosis_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(
        F.col("icd_version").desc(),
        F.col("icd_code").asc()
    )
)

primary_diagnosis = (
    diagnoses
    .filter(
        F.col("hadm_id").isNotNull()
        & (F.col("seq_num") == 1)
    )
    .join(
        dim_diagnosis.select(
            "diagnosis_key",
            "icd_code",
            "icd_version"
        ),
        on=["icd_code", "icd_version"],
        how="left"
    )
    .withColumn(
        "_rn",
        F.row_number().over(primary_diagnosis_window)
    )
    .filter(F.col("_rn") == 1)
    .select(
        "hadm_id",
        F.col("diagnosis_key").alias("primary_diagnosis_key")
    )
)

diagnosis_summary = (
    bridge_admission_diagnosis
    .groupBy("hadm_id")
    .agg(
        F.count("*").alias("diagnosis_count")
    )
    .join(
        primary_diagnosis,
        "hadm_id",
        "left"
    )
)

display(
    diagnosis_summary.agg(
        F.count("*").alias("admissions_with_diagnoses"),
        F.sum(
            F.when(
                F.col("primary_diagnosis_key").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_primary_diagnosis")
    )
)

In [0]:
# Build admission fact
# Grain: one row per hadm_id

fact_admission = (
    admissions.alias("a")

    # Patient context needed for admission-specific age
    .join(
        patients.select(
            "subject_id",
            "anchor_age",
            "anchor_year"
        ).alias("p"),
        F.col("a.subject_id") == F.col("p.subject_id"),
        "left"
    )

    # Admission-level summaries
    .join(
        transfer_summary.alias("t"),
        F.col("a.hadm_id") == F.col("t.hadm_id"),
        "left"
    )
    .join(
        service_summary.alias("s"),
        F.col("a.hadm_id") == F.col("s.hadm_id"),
        "left"
    )
    .join(
        icu_summary.alias("i"),
        F.col("a.hadm_id") == F.col("i.hadm_id"),
        "left"
    )
    .join(
        diagnosis_summary.alias("d"),
        F.col("a.hadm_id") == F.col("d.hadm_id"),
        "left"
    )
    .join(
        procedure_summary.alias("pr"),
        F.col("a.hadm_id") == F.col("pr.hadm_id"),
        "left"
    )

    # Patient dimension key
    .join(
        dim_patient
        .select("subject_id", "patient_key")
        .alias("dp"),
        F.col("a.subject_id") == F.col("dp.subject_id"),
        "left"
    )

    # Select admission grain before dimension lookups
    .select(
        F.col("a.hadm_id").alias("hadm_id"),
        F.col("a.subject_id").alias("subject_id"),
        F.col("dp.patient_key").alias("patient_key"),

        F.col("a.admittime").alias("admittime"),
        F.col("a.dischtime").alias("dischtime"),

        F.to_date("a.admittime").alias("admission_date"),
        F.to_date("a.dischtime").alias("discharge_date"),

        F.col("a.admission_type").alias("admission_type"),
        F.col("a.admission_location").alias("admission_location"),
        F.col("a.discharge_location").alias("discharge_location"),

        F.col("a.hospital_expire_flag")
        .cast("int")
        .alias("mortality_flag"),

        (
            F.col("p.anchor_age")
            + F.year("a.admittime")
            - F.col("p.anchor_year")
        ).alias("age_at_admission"),

        F.round(
            (
                F.col("a.dischtime").cast("long")
                - F.col("a.admittime").cast("long")
            ) / 86400.0,
            2
        ).alias("hospital_los_days"),

        F.coalesce(
            F.col("t.transfer_count"),
            F.lit(0)
        ).alias("transfer_count"),

        F.coalesce(
            F.col("t.distinct_careunit_count"),
            F.lit(0)
        ).alias("distinct_careunit_count"),

        F.coalesce(
            F.col("t.first_careunit"),
            F.lit("UNKNOWN")
        ).alias("first_careunit"),

        F.coalesce(
            F.col("t.last_careunit"),
            F.lit("UNKNOWN")
        ).alias("last_careunit"),

        F.col("s.initial_service").alias("initial_service"),
        F.col("s.final_service").alias("final_service"),

        F.coalesce(
            F.col("s.service_change_count"),
            F.lit(0)
        ).alias("service_change_count"),

        F.coalesce(
            F.col("i.icu_flag"),
            F.lit(0)
        ).alias("icu_flag"),

        F.coalesce(
            F.col("i.icu_stay_count"),
            F.lit(0)
        ).alias("icu_stay_count"),

        F.when(
            F.col("i.icu_flag").isNull(),
            F.lit(0.0)
        ).otherwise(
            F.col("i.total_icu_los_days")
        ).alias("total_icu_los_days"),

        F.when(
            F.col("i.icu_flag").isNull(),
            F.lit(True)
        ).otherwise(
            F.col("i.icu_los_complete")
        ).alias("icu_los_complete"),

        F.coalesce(
            F.col("d.diagnosis_count"),
            F.lit(0)
        ).alias("diagnosis_count"),

        F.col("d.primary_diagnosis_key")
        .alias("primary_diagnosis_key"),

        F.coalesce(
            F.col("pr.procedure_count"),
            F.lit(0)
        ).alias("procedure_count")
    )

    # Analytical bands
    .withColumn(
        "age_group",
        F.when(F.col("age_at_admission") < 40, "18-39")
         .when(F.col("age_at_admission") < 60, "40-59")
         .when(F.col("age_at_admission") < 80, "60-79")
         .otherwise("80+")
    )
    .withColumn(
        "los_band",
        F.when(F.col("hospital_los_days") < 1, "<1 day")
         .when(F.col("hospital_los_days") < 3, "1-3 days")
         .when(F.col("hospital_los_days") < 7, "3-7 days")
         .when(F.col("hospital_los_days") < 14, "7-14 days")
         .otherwise("14+ days")
    )

    # Date dimension keys
    .withColumn(
        "admission_date_key",
        F.date_format("admission_date", "yyyyMMdd").cast("int")
    )
    .withColumn(
        "discharge_date_key",
        F.date_format("discharge_date", "yyyyMMdd").cast("int")
    )

    # Careunit dimension keys
    .join(
        dim_careunit
        .select(
            F.col("careunit_name").alias("first_careunit"),
            F.col("careunit_key").alias("first_careunit_key")
        ),
        on="first_careunit",
        how="left"
    )
    .join(
        dim_careunit
        .select(
            F.col("careunit_name").alias("last_careunit"),
            F.col("careunit_key").alias("last_careunit_key")
        ),
        on="last_careunit",
        how="left"
    )

    # Service dimension keys
    .join(
        dim_service
        .select(
            F.col("service_code").alias("initial_service"),
            F.col("service_key").alias("initial_service_key")
        ),
        on="initial_service",
        how="left"
    )
    .join(
        dim_service
        .select(
            F.col("service_code").alias("final_service"),
            F.col("service_key").alias("final_service_key")
        ),
        on="final_service",
        how="left"
    )

    # Keep keys/measures in the fact; descriptions remain in dimensions
    .select(
        "hadm_id",
        "subject_id",

        "patient_key",
        "admission_date_key",
        "discharge_date_key",
        "first_careunit_key",
        "last_careunit_key",
        "initial_service_key",
        "final_service_key",
        "primary_diagnosis_key",

        "admittime",
        "dischtime",
        "admission_type",
        "admission_location",
        "discharge_location",

        "age_at_admission",
        "age_group",
        "hospital_los_days",
        "los_band",
        "mortality_flag",

        "transfer_count",
        "distinct_careunit_count",
        "service_change_count",

        "icu_flag",
        "icu_stay_count",
        "total_icu_los_days",
        "icu_los_complete",

        "diagnosis_count",
        "procedure_count"
    )
)

display(
    fact_admission.agg(
        F.count("*").alias("rows"),
        F.countDistinct("hadm_id").alias("unique_admissions"),
        F.countDistinct("subject_id").alias("unique_patients"),
        F.sum("icu_flag").alias("icu_admissions"),
        F.sum("mortality_flag").alias("in_hospital_deaths")
    )
)

In [0]:
# Validate admission fact foreign keys

fact_key_validation = fact_admission.agg(
    F.count("*").alias("rows"),

    F.sum(
        F.when(F.col("patient_key").isNull(), 1).otherwise(0)
    ).alias("missing_patient_key"),

    F.sum(
        F.when(F.col("admission_date_key").isNull(), 1).otherwise(0)
    ).alias("missing_admission_date_key"),

    F.sum(
        F.when(F.col("discharge_date_key").isNull(), 1).otherwise(0)
    ).alias("missing_discharge_date_key"),

    F.sum(
        F.when(F.col("first_careunit_key").isNull(), 1).otherwise(0)
    ).alias("missing_first_careunit_key"),

    F.sum(
        F.when(F.col("last_careunit_key").isNull(), 1).otherwise(0)
    ).alias("missing_last_careunit_key"),

    F.sum(
        F.when(F.col("initial_service_key").isNull(), 1).otherwise(0)
    ).alias("missing_initial_service_key"),

    F.sum(
        F.when(F.col("final_service_key").isNull(), 1).otherwise(0)
    ).alias("missing_final_service_key"),

    F.sum(
        F.when(F.col("primary_diagnosis_key").isNull(), 1).otherwise(0)
    ).alias("missing_primary_diagnosis_key")
)

display(fact_key_validation)

In [0]:
# Validate bridge grains before publishing

bridge_validation = spark.createDataFrame([
    {
        "bridge": "bridge_admission_diagnosis",
        "rows": bridge_admission_diagnosis.count(),
        "unique_rows": (
            bridge_admission_diagnosis
            .select(
                "hadm_id",
                "diagnosis_key",
                "seq_num"
            )
            .dropDuplicates()
            .count()
        )
    },
    {
        "bridge": "bridge_admission_procedure",
        "rows": bridge_admission_procedure.count(),
        "unique_rows": (
            bridge_admission_procedure
            .select(
                "hadm_id",
                "procedure_key",
                "seq_num"
            )
            .dropDuplicates()
            .count()
        )
    }
])

bridge_validation = (
    bridge_validation
    .withColumn(
        "status",
        F.when(
            F.col("rows") == F.col("unique_rows"),
            "PASS"
        ).otherwise("CHECK")
    )
)

display(bridge_validation)

In [0]:
# Publish admission fact and bridges to Gold

admission_star_tables = {
    "fact_admission": fact_admission,
    "bridge_admission_diagnosis": bridge_admission_diagnosis,
    "bridge_admission_procedure": bridge_admission_procedure
}

for table_name, df in admission_star_tables.items():

    output_df = df.withColumn(
        "_gold_timestamp",
        F.current_timestamp()
    )

    (
        output_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.{table_name}")
    )

    print(f"Published: {GOLD_SCHEMA}.{table_name}")

In [0]:
# Final persisted validation

persisted_fact = spark.table(f"{GOLD_SCHEMA}.fact_admission")
persisted_diag_bridge = spark.table(
    f"{GOLD_SCHEMA}.bridge_admission_diagnosis"
)
persisted_proc_bridge = spark.table(
    f"{GOLD_SCHEMA}.bridge_admission_procedure"
)

validation_rows = [
    {
        "table": "fact_admission",
        "rows": persisted_fact.count(),
        "unique_grain_rows": (
            persisted_fact
            .select("hadm_id")
            .dropDuplicates()
            .count()
        )
    },
    {
        "table": "bridge_admission_diagnosis",
        "rows": persisted_diag_bridge.count(),
        "unique_grain_rows": (
            persisted_diag_bridge
            .select(
                "hadm_id",
                "diagnosis_key",
                "seq_num"
            )
            .dropDuplicates()
            .count()
        )
    },
    {
        "table": "bridge_admission_procedure",
        "rows": persisted_proc_bridge.count(),
        "unique_grain_rows": (
            persisted_proc_bridge
            .select(
                "hadm_id",
                "procedure_key",
                "seq_num"
            )
            .dropDuplicates()
            .count()
        )
    }
]

validation_df = (
    spark.createDataFrame(validation_rows)
    .withColumn(
        "status",
        F.when(
            F.col("rows") == F.col("unique_grain_rows"),
            "PASS"
        ).otherwise("CHECK")
    )
)

display(validation_df)
